# StarGAN
 
**Date**: 2025/02/04

## 1. Introduction

In **image-to-image translation**, our goal is to learn a function (often parametrized by a neural network, e.g. a **generator**) that translates an image from one domain $X$ (e.g., images of male faces) to another domain $Y$ (e.g., images of female faces). Early successes in this domain have been demonstrated by models such as **Pix2Pix** (paired training) and **CycleGAN** (unpaired training). However:

- **CycleGAN** is limited to learning a mapping between only two domains, $X\leftrightarrow Y$.
- **StarGAN v1** extends CycleGAN to support multi-domain translation in a single unified framework.
- **StarGAN v2** further improves upon StarGAN v1 by enabling **diverse** (multi-modal) outputs within each domain, all in a single unified model for multiple domains.

In this notebook, we will describe the **mathematical formulation** and **theoretical concepts** behind **StarGAN v1** and **StarGAN v2**, and how **StarGAN v1** builds upon **CycleGAN**. No code is included here; we focus on the conceptual theory and advanced mathematical details.


## 2. StarGAN v1: Multi-Domain Image-to-Image Translation

### 2.1 Motivation

<p align="center">
  <img src="./imgs/stargan-v1_fig-2.png" alt="drawing" width="700"/>
</p>


While CycleGAN can translate between **two** domains $X\leftrightarrow Y$, many real-world tasks require more than two domains. For example, consider the **CelebA** dataset with attributes {young, old, male, female, blond, brown, black hair color, etc.}. If we want a single model to handle translations across $K$ different domains (hair color + age + gender, etc.), the naive approach with CycleGAN would require $K(K-1)$ separate generators, which is highly inefficient.

**StarGAN v1** solves this issue by:

1. Using a **single generator** $G$ for all domains.
2. Conditioning the generator and discriminator on **domain labels**.

Hence, StarGAN v1 unifies multi-domain mappings (say, from domain $i$ to domain $j$) with only one generator and one discriminator.

### 2.2 Architecture and Objective

<p align="center">
  <img src="./imgs/stargan-v1_fig-3.png" alt="drawing" width="900"/>
</p>

#### (a) Unified Generator with Domain Information

Let there be $K$ domains labeled by $\{1,2,\dots,K\}$. Given an input image $x$ from domain $i$ and a **desired** domain label $c\in\{1,\dots,K\}$, the generator of StarGAN v1 is:

$$
G(x, c)\;\to\;\hat{x},
$$

where $\hat{x}$ is the translated image that should lie in domain $c$. Traditionally, StarGAN v1 represents the domain $c$ as a **one-hot vector** or a **binary attribute vector**. The generator **concatenates** or **injects** $c$ at some layer (e.g., via channel-wise concatenation or AdaIN-like normalization in some variations).

**Key difference from CycleGAN**: StarGAN v1’s generator is multi-domain. Instead of two separate generators $(G,F)$, StarGAN v1 has one generator $G$ that uses a domain label for controlling the *direction* and *type* of translation.

#### (b) Multi-Task Discriminator

Instead of separate discriminators $(D_X, D_Y)$ for each domain, StarGAN v1 has a **single multi-task discriminator** $D$, whose architecture branches near the output to classify:

- **Real vs. Fake** (i.e., is this image generated or authentic?)
- **Domain classification** (which domain does this image belong to?).

#### (c) Loss Functions in StarGAN v1

1. **Adversarial Loss** (WGAN-GP or similar) encourages the generator to produce realistic images for the target domain.
2. **Domain Classification Loss**: 
   - **Real-image classification** ($\mathcal{L}_{\mathrm{cls}}^{r}$), optimized **only by $D$**:
    $$
    \mathcal{L}_{\mathrm{cls}}^{r}(D) 
    \;=\;
    \mathbb{E}_{(x,\,c')}\Bigl[
        -\log D_{\mathrm{cls}}\bigl(c'\,\big|\,x\bigr)
    \Bigr],
    $$
    where $x$ is a real image from the domain $c'$.  
    This forces $D$ to classify each **real** image correctly to its **true** domain $c'$.
   - **Fake-image classification** ($\mathcal{L}_{\mathrm{cls}}^{f}$), optimized **only by $G$**:
    $$
    \mathcal{L}_{\mathrm{cls}}^{f}(G) 
    \;=\;
    \mathbb{E}_{(x,\,c)}\Bigl[
        -\log D_{\mathrm{cls}}\bigl(c\,\big|\,G(x,c)\bigr)
    \Bigr].
    $$
    Here, $x$ is an input image from *any* domain, and $c$ is a **target** domain label. We translate $x\to G(x,c)$, and want $D_{\mathrm{cls}}$ to classify the translated image as domain $c$.
3. **Cycle Consistency Loss**: A variant of the cycle consistency ensures the image does not lose key identity characteristics.  
   StarGAN v1 reuses the idea from CycleGAN: if we first translate $x$ from domain $i$ to domain $j$, then we attempt to translate it back from $j\to i$, the recovered image must be close to the original.

$$
\mathcal{L}_{\mathrm{rec}}(G) 
\;=\;
\mathbb{E}_{x, c, c'} 
\left\lVert 
x - G\bigl( G(x, c), c' \bigr)\right\rVert_1,
$$
where $c'$ is the original domain label of $x$.

Putting all components together, StarGAN v1 solves:

$$
\min_{G}\,\max_{D}\quad
\mathcal{L}_{\mathrm{adv}}
\;+\;
\lambda_{\mathrm{cls}}
\bigl[
  \mathcal{L}_{\mathrm{cls}}^{r}(D) \;+\; 
  \mathcal{L}_{\mathrm{cls}}^{f}(G)
\bigr]
\;+\;
\lambda_{\mathrm{rec}}\,
\mathcal{L}_{\mathrm{rec}}(G),
$$

where $\lambda_{\mathrm{cls}}$ and $\lambda_{\mathrm{rec}}$ are hyperparameters weighting the domain classification and reconstruction losses.

#### (d) How StarGAN v1 Extends CycleGAN

- **Single model**: Instead of two separate generators (CycleGAN) for two domains, StarGAN v1 has a single generator that can handle $K$ domains using domain labels. 
- **Multi-domain**: In place of $\mathcal{L}_{\mathrm{GAN}}(G, D)$ for just $X\leftrightarrow Y$, we have an adversarial loss that considers $(x, c)\to \hat{x}$ for any domain combination. A domain classifier within $D$ enables domain control.
- **Unified approach**: CycleGAN’s cycle-consistency concept is **retained**, but StarGAN v1 adds domain classification to handle multiple attributes or domain conditions.

### 2.3 Limitations of StarGAN v1

Although StarGAN v1 successfully handles multiple domains in a single generator, it outputs a single deterministic image per **(input image, target domain label)** pair. That is, the domain label vector $\mathbf{c}\in \{0,1\}^K$ is fixed for each translation, so there is no built-in notion of multi-modal or diverse outputs **within** a single domain. For instance, if we fix “target domain = ‘female’” in a face-attribute translation, StarGAN v1 will produce (roughly) the **same** output each time.


## 3. StarGAN v2: Improved Diversity across Multiple Domains

### 3.1 Key Goal: Multi-Domain and Multi-Modal

StarGAN v2, proposed in [**StarGAN v2: Diverse Image Synthesis for Multiple Domains** (Choi et al., 2020)](https://arxiv.org/abs/1912.01865), **retains** the single-generator approach for multi-domain translation but adds the capability for **diverse** or **multi-modal** outputs **within** each domain. This effectively solves two major challenges at once:

1. **Scalability** over multiple domains.
2. **Diverse** (multi-modal) image generation within each domain.

### 3.2 Architecture Components

<p align="center">
  <img src="./imgs/stargan-v2_fig-2.png" alt="drawing" width="900"/>
</p>

Let us denote:
- $G$: A **single** generator that takes an input image $x$ and a **domain-specific style** $\mathbf{s}$, generating a translated image $\hat{x}$.
- $F$: A **mapping network** that encodes a random latent code $\mathbf{z}$ into a **domain-specific style** $\mathbf{s}=F_y(\mathbf{z})$. This style $\mathbf{s}$ must represent a specific domain $y$.
- $E$: A **style encoder** that extracts a style code $\mathbf{s}=E_y(x)$ from any reference image $x$ of domain $y$.
- $D$: A **multi-task discriminator**, similar in spirit to StarGAN v1, that classifies real vs. fake and the domain label.

Hence, we have:

$$
\hat{x} = G\bigl(x, \mathbf{s}\bigr)
\quad\text{where}\quad
\mathbf{s} = 
\begin{cases}
F_y(\mathbf{z}), & \text{(latent-guided)} \\
E_y(x_{\mathrm{ref}}), & \text{(reference-guided)}
\end{cases}
$$

Depending on usage, we can either:
- **Sample** random noise $\mathbf{z}\sim\mathcal{N}(0, I)$ and map it to $\mathbf{s}$ for **diverse** or **multi-modal** generation.
- **Extract** the style from a reference image $x_{\mathrm{ref}}$. This allows reference-guided translation, reproducing the style from $x_{\mathrm{ref}}$.

### 3.3 Loss Functions

StarGAN v2 extends the standard adversarial and cycle-like constraints from v1, plus new components to encourage **style diversity**.

1. **Adversarial Loss**:  
   Let $y$ be the target domain. The generator $G$ uses $\mathbf{s}=F_y(\mathbf{z})$. The discriminator $D$ tries to determine if $\hat{x} = G(x,\mathbf{s})$ is real or fake for domain $y$. For instance, using the non-saturating GAN loss:

   $$
   \mathcal{L}_{\mathrm{adv}} =
   \mathbb{E}_{x,y}\Bigl[ 
       \log D_y(x)
   \Bigr]
   \;+\;
   \mathbb{E}_{x,y,\mathbf{z}}\Bigl[
       \log \bigl(1 - D_y\bigl(G(x,F_y(\mathbf{z}))\bigr)\bigr)
   \Bigr].
   $$

2. **Style Reconstruction Loss**:  
   After $G$ synthesizes an image $\hat{x}=G(x,\mathbf{s})$, the style encoder $E$ should be able to recover the same style $\mathbf{s}$. I.e.:

   $$
   \mathcal{L}_{\mathrm{sty}} 
   = 
   \mathbb{E}_{x,y,\mathbf{z}}\bigl\|
       \mathbf{s} - E_y\bigl( G(x,\mathbf{s}) \bigr)
   \bigr\|_1,
   $$
   where $\mathbf{s}=F_y(\mathbf{z})$. This ensures the generated image truly reflects the style code $\mathbf{s}$.

3. **Diversity Sensitive Loss** (a.k.a. multi-modal or style diversification):
   
   To explicitly encourage diverse outputs given different latent codes $\mathbf{z}_1,\mathbf{z}_2$, StarGAN v2 includes:

   $$
   \mathcal{L}_{\mathrm{ds}}
   =
   \mathbb{E}_{x,y,\mathbf{z}_1,\mathbf{z}_2}\!\Bigl[
   \bigl\|
     G\bigl(x, F_y(\mathbf{z}_1)\bigr)
     \;-\;
     G\bigl(x, F_y(\mathbf{z}_2)\bigr)
   \bigr\|_1
   \Bigr],
   $$
   (possibly normalized by $\|\mathbf{z}_1 - \mathbf{z}_2\|$ or handled carefully to avoid instabilities). Maximizing $\mathcal{L}_{\mathrm{ds}}$ ensures the generator produces **distinct** images for distinct latent codes. In practice, the objective is appended with a negative sign in the total cost (i.e., we want to **maximize** diversity or **minimize** $-\mathcal{L}_{\mathrm{ds}}$).

4. **Source Consistency** (a light form of cycle or identity):
   
   A simpler cycle-like constraint is used so that the generator $G$ does not lose the essential structure or identity of $x$. StarGAN v2 often enforces:

   $$
   \mathcal{L}_{\mathrm{cyc}}(G)
   =
   \mathbb{E}_{x,y,\mathbf{z}}\Bigl[
     \bigl\|
       x - G\bigl( G(x,F_y(\mathbf{z})),\, E_{\text{orig}}(x)\bigr)
     \bigr\|_1
   \Bigr],
   $$
   where $E_{\text{orig}}$ extracts the original domain style from $x$. The exact form can vary, but the main idea is to maintain *source* content while modifying only style.

### 3.4 Final Objective

The combined objective for StarGAN v2 typically is:

$$
\min_{G, F, E} \max_{D}
\quad
\mathcal{L}_{\mathrm{adv}} 
\;+\;
\lambda_{\mathrm{sty}}\,\mathcal{L}_{\mathrm{sty}}
\;-\;
\lambda_{\mathrm{ds}}\,\mathcal{L}_{\mathrm{ds}}
\;+\;
\lambda_{\mathrm{cyc}}\,\mathcal{L}_{\mathrm{cyc}},
$$

where $\lambda_{\mathrm{sty}}, \lambda_{\mathrm{ds}}, \lambda_{\mathrm{cyc}}$ balance the style reconstruction, diversity, and source consistency terms.

### 3.5 Improvements over StarGAN v1

1. **Style injection** for multi-modal outputs:  
   StarGAN v1 uses domain labels as input (e.g., a one-hot vector). StarGAN v2 replaces the domain label with a **style code** $\mathbf{s}$ that can vary among infinitely many styles **within** the same domain.

2. **Reference-guided** or **latent-guided**:  
   - **Latent-guided**: Sample $\mathbf{z}\sim\mathcal{N}(0,I)$, map to $\mathbf{s}=F_y(\mathbf{z})$. This yields diverse random styles.
   - **Reference-guided**: For a reference image $x_{\mathrm{ref}}$ in domain $y$, extract style $\mathbf{s}=E_y(x_{\mathrm{ref}})$. Then produce $\hat{x}=G(x,\mathbf{s})$. This can replicate the style of a specific reference.

3. **Higher-quality outputs**:  
   Empirically, StarGAN v2 is known to achieve better FID (Fréchet Inception Distance) with more realistic images across multiple domains than StarGAN v1, thanks to more flexible style representation and multi-task learning synergy.

4. **Scalable**:  
   Like StarGAN v1, StarGAN v2 remains a single unified model for all domains. The multi-branch architecture in the mapping network $F$ and style encoder $E$ easily scales as the number of domains grows.


## 4. Conclusions

### 4.1 Summary of Key Points

1. **CycleGAN** provided the concept of **cycle consistency** to learn unpaired image-to-image mappings for two domains.
2. **StarGAN v1** extended CycleGAN to a **single generator** that can handle multiple domains by conditioning on a domain label.  
   $\quad\Rightarrow$ *Limitation:* a single output style per domain label.
3. **StarGAN v2** introduced a **style space** for each domain, enabling **diverse** or **multi-modal** outputs within a single domain. Through the mapping network and style encoder, StarGAN v2 synthesizes high-quality, multi-domain, multi-modal image translations in a unified framework.